# Embedding space overlap analysis  (2014)

In [1]:
import pandas as pd
import numpy as np
import random as rd
from sklearn.neighbors import NearestNeighbors
from typing import List
from math import log2

## 1. Read embedding files

In [3]:
title_df = pd.read_csv('../../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.title', sep='\t')
title_df.head(5)

,ent_id:token,ent_emb:float_seq
0,0,0.059633706 0.06158079 0.085539125 0.09338283 ...
1,1,0.019596875 0.13992491 -0.016762894 0.04142571...
2,3,0.037360948 0.09006915 0.030246628 0.022775028...
3,2,0.0773414 0.12009948 0.01776859 -0.005860431 0...
4,4,0.027500087 0.0679503 0.06407129 0.026295394 0...


In [4]:
image_df = pd.read_csv('../../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.image', sep='\t')
image_df.head(5)

,img_id:token,img_emb:float_seq
0,0,0.014428207 -0.01609465 0.03294648 0.010992038...
1,1,0.013374746 -0.0040851366 0.017846484 0.009146...
2,3,0.025485406 0.044892143 0.06540884 0.07273897 ...
3,2,0.0927549 0.0003234906 0.0068842433 0.05643998...
4,4,0.026845569 -0.0059881625 0.06385405 -0.019091...


## 2. Preprocess embeddings

In [5]:
title_df.rename(columns={"ent_id:token": "id", "ent_emb:float_seq": "emb"}, inplace=True)
title_df["emb"] = title_df["emb"].apply(lambda x: np.fromstring(x, dtype=np.float32, sep=' '))
title_df.head(5)

,id,emb
0,0,"[0.059633706, 0.06158079, 0.085539125, 0.09338..."
1,1,"[0.019596875, 0.13992491, -0.016762894, 0.0414..."
2,3,"[0.037360948, 0.09006915, 0.030246628, 0.02277..."
3,2,"[0.0773414, 0.12009948, 0.01776859, -0.0058604..."
4,4,"[0.027500087, 0.0679503, 0.06407129, 0.0262953..."


In [6]:
image_df.rename(columns={"img_id:token": "id", "img_emb:float_seq": "emb"}, inplace=True)
image_df["emb"] = image_df["emb"].apply(lambda x: np.fromstring(x, dtype=np.float32, sep=' '))
image_df.head(5)

,id,emb
0,0,"[0.014428207, -0.01609465, 0.03294648, 0.01099..."
1,1,"[0.013374746, -0.0040851366, 0.017846484, 0.00..."
2,3,"[0.025485406, 0.044892143, 0.06540884, 0.07273..."
3,2,"[0.0927549, 0.0003234906, 0.0068842433, 0.0564..."
4,4,"[0.026845569, -0.0059881625, 0.06385405, -0.01..."


## 3. Fit KNN with static embeddings

In [7]:
X = np.vstack(title_df["emb"].values)
title_knn = NearestNeighbors(n_neighbors=100, metric='cosine')
title_knn.fit(X)

,n_neighbors,100
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [8]:
Y = np.vstack(image_df["emb"].values)
image_knn = NearestNeighbors(n_neighbors=100, metric='cosine')
image_knn.fit(Y)

,n_neighbors,100
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


## 4. Metrics

In [9]:
def iou_at_k(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    set_a = set(neighbors_a[:k])
    set_b = set(neighbors_b[:k])
    return len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 0.0

def recall_at_k(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    set_a = set(neighbors_a[:k])
    set_b = set(neighbors_b[:k])
    return len(set_a & set_b) / len(set_a) if set_a else 0.0

def dcg_at_k(relevance: List[int], k: int) -> float:
    return sum(rel / log2(idx + 2) for idx, rel in enumerate(relevance[:k]))

def ndcg_between(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    ideal_relevance = [k - i for i in range(min(k, len(neighbors_a)))]
    position_map = {cid: i for i, cid in enumerate(neighbors_a[:k])}
    relevance_b = []
    for cid in neighbors_b[:k]:
        if cid in position_map:
            relevance_b.append(k - position_map[cid])
        else:
            relevance_b.append(0)
    dcg = dcg_at_k(relevance_b, k)
    ideal_dcg = dcg_at_k(ideal_relevance, k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

## 5. Analysis

In [10]:
from tqdm import tqdm

original_ids = list(image_df['id'])
ids = range(len(original_ids))
BATCH_SIZE = 1000
iou, recall, ndcg = 0.0, 0.0, 0.0

for L in tqdm(range(0, len(ids), BATCH_SIZE), unit="batch"):
    R = min(L + BATCH_SIZE, len(ids))
    ids_slice = ids[L:R]

    title_distance, title_nearest_ids = title_knn.kneighbors([X[idx] for idx in ids_slice])
    image_distance, image_nearest_ids = image_knn.kneighbors([Y[idx] for idx in ids_slice])

    for t_ids, i_ids in zip(title_nearest_ids, image_nearest_ids):
        iou += iou_at_k(t_ids, i_ids, 100)
        recall += recall_at_k(t_ids, i_ids, 100)
        ndcg += ndcg_between(t_ids, i_ids, 100)

iou = iou * 100 / len(original_ids)
recall = recall * 100 / len(original_ids)
ndcg = ndcg * 100 / len(original_ids)

print(f"IOU@100: {iou:.2f}%, Recall@100: {recall:.2f}%, NDCG@100: {ndcg:.2f}%")

100%|██████████| 19/19 [00:04<00:00,  3.80batch/s]

IOU@100: 6.64%, Recall@100: 11.54%, NDCG@100: 21.26%
